# IAD (200 Epochs KIS + 100 Epochs KAS) — 3 Seeds (2345, 42, 1234)

Trains two-stage IAD (`Arm_IAD_200_100`) under a matched 300-epoch total budget per seed: Stage 1 (epochs 1..200) runs KIS (`0.8 * L_SGS + 0.2 * L_KIS`, `flat` CPD weights), and Stage 2 (epochs 201..300) runs KAS (`L_SGS + 0.1 * (L_RFC + L_CSS)`) with registered `IADKasHead` convolutions (`pos` 1x1 and `neg` 3x3). Permanent checkpoints are saved at epoch 200 and epoch 300.

- **Seeds**: `[2345, 42, 1234]` (3 seeds, run sequentially in this notebook or across sessions).
- **Total Budget**: `300` epochs per seed on all 18 native BTCV training cases (`2,211` slices).
- **Evaluation Schedule**: Final epoch only (`--eval_freq 0`) on the 12 official 3D BTCV test volumes (`data/btcv/test_vol_h5`). Intermediate warmup/Stage-1 checkpoints do not run 3D volume evaluation.
- **Input Modes**:
  - `INPUT_MODE = "upload"` (default): Upload `iad_cba_v2_code.zip`, `btcv_data.zip`, and `epoch_299.pth` into `/content/`. SAM ViT-L base weights (`sam_vit_l_0b3195.pth`) download automatically if not uploaded.
  - `INPUT_MODE = "drive"`: Mounts Google Drive (`MyDrive/IAD`), reads inputs from Drive, and syncs checkpoints/metrics to `MyDrive/IAD/main/iad_200_100*` every 5 epochs.
- **Disconnect Recovery**: Re-running cells 1–4 skips any seed that already has `eval_final.json` and resumes any interrupted seed from its latest checkpoint (`latest_checkpoint.pth`).

In [ ]:
# 1. CONFIG
INPUT_MODE = "upload"  # "upload" for /content files; "drive" to mount Google Drive.
RUN = "iad"
SEEDS = [2345, 42, 1234]
EPOCHS = 300
WORK = "/content/IAD"
RUNS = "/content/runs/main"
DRIVE_ROOT = "/content/drive/MyDrive/IAD"

In [ ]:
# 2. SETUP: use uploaded files in /content, or switch INPUT_MODE to drive.
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

if INPUT_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    INPUT = Path(DRIVE_ROOT)
    MAIN_DRIVE = str(INPUT / "main")
elif INPUT_MODE == "upload":
    INPUT = Path("/content")
    MAIN_DRIVE = str(Path(DRIVE_ROOT) / "main") if os.path.isdir(DRIVE_ROOT) else None
else:
    raise ValueError("INPUT_MODE must be 'upload' or 'drive'")

work = Path(WORK)
work.mkdir(parents=True, exist_ok=True)
for name in ("iad_cba_v2_code.zip", "btcv_data.zip"):
    if name == "btcv_data.zip":
        npz_dir = work / "data" / "btcv" / "train_npz"
        h5_dir = work / "data" / "btcv" / "test_vol_h5"
        if npz_dir.is_dir() and h5_dir.is_dir() and len(list(npz_dir.glob("*.npz"))) >= 2211 and len(list(h5_dir.glob("*.h5"))) >= 12:
            print("BTCV data already extracted in", work)
            continue
    archive = INPUT / name
    if not archive.is_file() or not zipfile.is_zipfile(archive):
        raise RuntimeError(f"Missing or incomplete ZIP: {archive}. Finish uploading the file before running setup.")
    with zipfile.ZipFile(archive) as z:
        bad = z.testzip()
        if bad:
            raise RuntimeError(f"Corrupt ZIP entry: {bad} in {archive}")
        z.extractall(work)

for _mod in list(sys.modules):
    if _mod in ("main_common", "run_tuning") or _mod.startswith(("experiments.", "datasets.", "Synapse_Tool_KD.")):
        sys.modules.pop(_mod, None)

tea_src = INPUT / "epoch_299.pth"
if not tea_src.is_file():
    tea_src = INPUT / "teacher_weights" / "epoch_299.pth"
if not tea_src.is_file():
    raise FileNotFoundError(f"Upload epoch_299.pth into {INPUT}")
tea_dst = work / "teacher_weights" / "epoch_299.pth"
tea_dst.parent.mkdir(parents=True, exist_ok=True)
if not tea_dst.is_file() or tea_dst.stat().st_size != tea_src.stat().st_size:
    shutil.copy2(tea_src, tea_dst)
sam_dst = work / "data" / "weights" / "sam_vit_l_0b3195.pth"
sam_dst.parent.mkdir(parents=True, exist_ok=True)
sam_src = INPUT / "data" / "weights" / sam_dst.name
if not sam_dst.is_file() or sam_dst.stat().st_size < 1_200_000_000:
    if sam_src.is_file():
        shutil.copy2(sam_src, sam_dst)
    else:
        subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(sam_dst),
                        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth"], check=True)
assert sam_dst.stat().st_size >= 1_200_000_000, "Incomplete SAM base weights"
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
if f"{WORK}/scripts" not in sys.path:
    sys.path.insert(0, f"{WORK}/scripts")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements_colab.txt"], check=True)

# Freeze the canonical v5 parameters for all seeds; do not read a changing tuning state.
STATE = str(work / "multiseed_v5_state.json")
Path(STATE).write_text(json.dumps({"lambda_a": 0.02, "tau": 0.1, "m": 0.2, "r": 2, "beta": 0.5}))

def run_command(command):
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f"Training command failed with exit code {code}")

def command_for(seed, dry_run=False):
    cmd = [sys.executable, f"scripts/run_main_{RUN}.py", "--runs_dir", RUNS,
           "--epochs", str(EPOCHS), "--seed", str(seed), "--eval_freq", "0"]
    if MAIN_DRIVE:
        cmd += ["--drive_dir", MAIN_DRIVE]
    if RUN == "ours":
        cmd += ["--arm", "v5", "--state_json", STATE, "--make_warmup"]
    if dry_run:
        cmd.append("--dry_run")
    return cmd
print("Input mode:", INPUT_MODE, "| method:", RUN, "| seeds:", SEEDS, "| checkpoint backup:", MAIN_DRIVE or "local /content only")

In [ ]:
# 3. VERIFY: environment, bundled launcher, data, and seed-isolated dry runs across all 3 seeds.
import importlib, torch
assert torch.cuda.is_available(), "Select a GPU runtime in Colab first"
print("Torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
for module in ("icecream", "cv2", "einops", "medpy", "h5py", "timm", "scipy"):
    importlib.import_module(module)
import datasets.dataset_synapse as ds
assert Path(ds.__file__).resolve().is_relative_to(Path(WORK).resolve()), ds.__file__
assert (work / "data/lists_Synapse/train.txt").is_file()
assert (work / "data/btcv/test_vol_h5").is_dir()
import main_common as mc
assert mc.seed_name("warmup", 42) == "warmup_s42", "Old code ZIP: upload the rebuilt bundle"
assert {"kis", "iad", "lskd", "sinkd", "dist_plus"} <= set(mc.KIND_DIRS), "Old code ZIP: missing 5-method support"
from experiments.cba_v2.losses.cba_v2_loss import CBAv2Loss
_v5_check = CBAv2Loss(error_mode="ce_hard", error_gating_type="none")
assert (
    (not _v5_check.fp_by_pred_class)
    and (not _v5_check.fp_exclude_boundary)
    and (not _v5_check.empty_slice_suppression)
    and _v5_check.align_ce_hard_samples
), (
    "Code ZIP still has Solution 1+2 enabled by default in CBAv2Loss; upload/extract the reverted iad_cba_v2_code.zip"
)
del _v5_check
for seed in SEEDS:
    if RUN == "ours":
        _job = mc.job_for("ours", "v5", mc.rt.DEFAULT_STATE, EPOCHS, seed=seed)
        mc.archive_sol12_run_if_needed(_job, RUNS, MAIN_DRIVE)
        if MAIN_DRIVE and not mc.rt.is_done(_job, RUNS, MAIN_DRIVE) and not mc.rt.has_own_checkpoint(_job, RUNS, MAIN_DRIVE):
            local_wck = Path(RUNS) / mc.seed_name("warmup", seed) / "Arm_V1_KIS" / "checkpoint_epoch_020.pth"
            drive_wck = mc.find_warmup_ckpt(RUNS, MAIN_DRIVE, seed)
            if drive_wck and not local_wck.is_file():
                local_wck.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(drive_wck, local_wck)
                print(f"Staged existing Drive warmup for seed {seed}: {local_wck}", flush=True)
    run_command(command_for(seed, dry_run=True))

In [ ]:
# 4. RUN: train all 3 seeds (2345, 42, 1234) for 300 total epochs per seed.
# Final 12-volume 3D BTCV evaluation runs only at epoch 300 (--eval_freq 0).
# Re-running after a disconnect skips completed seeds and resumes any interrupted seed from its latest checkpoint.
for seed in SEEDS:
    if RUN == "ours":
        _job = mc.job_for("ours", "v5", mc.rt.DEFAULT_STATE, EPOCHS, seed=seed)
        mc.archive_sol12_run_if_needed(_job, RUNS, MAIN_DRIVE)
        uploaded = Path(f"/content/warmup_s{seed}.pth")
        if not uploaded.is_file() and seed == 2345:
            uploaded = Path("/content/warmup.pth")
        if uploaded.is_file():
            ckpt = torch.load(uploaded, map_location="cpu", weights_only=False)
            saved_args = ckpt.get("args", {})
            assert saved_args.get("seed") == seed, f"Uploaded warmup seed mismatch: {saved_args.get('seed')} != {seed}"
            assert ckpt.get("epoch") == 20, "Uploaded warmup must be at epoch 20"
            del ckpt
            dest = Path(RUNS) / mc.seed_name("warmup", seed) / "Arm_V1_KIS" / "checkpoint_epoch_020.pth"
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(uploaded, dest)
            print(f"Imported uploaded warmup for seed {seed}: {dest}", flush=True)
        elif MAIN_DRIVE and not mc.rt.is_done(_job, RUNS, MAIN_DRIVE) and not mc.rt.has_own_checkpoint(_job, RUNS, MAIN_DRIVE):
            local_wck = Path(RUNS) / mc.seed_name("warmup", seed) / "Arm_V1_KIS" / "checkpoint_epoch_020.pth"
            drive_wck = mc.find_warmup_ckpt(RUNS, MAIN_DRIVE, seed)
            if drive_wck and not local_wck.is_file():
                local_wck.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(drive_wck, local_wck)
                print(f"Staged existing Drive warmup for seed {seed}: {local_wck}", flush=True)
    run_command(command_for(seed))

In [ ]:
# 5. STATUS & 3-SEED SUMMARY: report each seed (Raw & Post-LCC) and aggregate Mean ± Std across completed seeds.
import csv, math
BASE_NAME = "ours_v5" if RUN == "ours" else mc.KIND_DIRS[RUN]
ARM_NAME = "Arm_V5_Dual_CEHard" if RUN == "ours" else mc.rt.ARM_DIRS[RUN]

dice_raw_vals, hd95_raw_vals, nsd_raw_vals = [], [], []
dice_lcc_vals, hd95_lcc_vals, nsd_lcc_vals = [], [], []
per_seed_records = {}
for seed in SEEDS:
    name = mc.seed_name(BASE_NAME, seed)
    local = Path(RUNS) / name / ARM_NAME
    mirror = Path(MAIN_DRIVE) / name if MAIN_DRIVE else local
    directory = local if (local / "metrics.csv").exists() else mirror
    metric_path = directory / "metrics.csv"
    if metric_path.is_file():
        with metric_path.open(encoding="utf-8") as handle:
            rows = list(csv.DictReader(handle))
        if rows:
            last = rows[-1]
            print(f"[{name}] epoch={last.get('epoch')} stage={last.get('stage', ARM_NAME)} loss={last.get('loss_total')} sec={last.get('time_sec')}")
    final = directory / "eval_final.json"
    if final.is_file():
        overall = json.loads(final.read_text(encoding="utf-8"))["overall"]
        dr = float(overall["primary_dice_raw_macro"]) * 100.0
        hr = float(overall["primary_hd95_raw_valid_macro"])
        nr = float(overall["primary_nsd_raw_macro"]) * 100.0
        dl = float(overall.get("primary_dice_lcc_macro", overall["primary_dice_raw_macro"])) * 100.0
        hl = float(overall.get("primary_hd95_lcc_valid_macro", overall["primary_hd95_raw_valid_macro"]))
        nl = float(overall.get("primary_nsd_lcc_macro", overall["primary_nsd_raw_macro"])) * 100.0
        dice_raw_vals.append(dr); hd95_raw_vals.append(hr); nsd_raw_vals.append(nr)
        dice_lcc_vals.append(dl); hd95_lcc_vals.append(hl); nsd_lcc_vals.append(nl)
        per_seed_records[str(seed)] = {
            "dice_raw_pct": dr, "hd95_raw": hr, "nsd_raw_pct": nr,
            "dice_lcc_pct": dl, "hd95_lcc": hl, "nsd_lcc_pct": nl,
        }
        print(f"  -> FINAL (seed {seed}) RAW: Dice={dr:.2f}% | HD95={hr:.2f} | NSD={nr:.2f}%")
        print(f"  -> FINAL (seed {seed}) LCC: Dice={dl:.2f}% | HD95={hl:.2f} | NSD={nl:.2f}%")

if dice_raw_vals:
    def mean_std(xs):
        m = sum(xs) / len(xs)
        s = math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs)) if len(xs) > 1 else 0.0
        return m, s
    drm, drs = mean_std(dice_raw_vals); hrm, hrs = mean_std(hd95_raw_vals); nrm, nrs = mean_std(nsd_raw_vals)
    dlm, dls = mean_std(dice_lcc_vals); hlm, hls = mean_std(hd95_lcc_vals); nlm, nls = mean_std(nsd_lcc_vals)
    print(f"\n=== {RUN.upper()} ({len(dice_raw_vals)}/{len(SEEDS)} seeds complete) ===")
    print(f"RAW -> Dice (%): {drm:.2f} ± {drs:.2f} | HD95: {hrm:.2f} ± {hrs:.2f} | NSD (%): {nrm:.2f} ± {nrs:.2f}")
    print(f"LCC -> Dice (%): {dlm:.2f} ± {dls:.2f} | HD95: {hlm:.2f} ± {hls:.2f} | NSD (%): {nlm:.2f} ± {nls:.2f}")
    summary_payload = {
        "method": RUN,
        "arm": ARM_NAME,
        "seeds_completed": [int(s) for s in per_seed_records],
        "per_seed": per_seed_records,
        "aggregate_raw": {"dice_mean": drm, "dice_std": drs, "hd95_mean": hrm, "hd95_std": hrs, "nsd_mean": nrm, "nsd_std": nrs},
        "aggregate_lcc": {"dice_mean": dlm, "dice_std": dls, "hd95_mean": hlm, "hd95_std": hls, "nsd_mean": nlm, "nsd_std": nls},
    }
    summary_path = Path(RUNS) / f"summary_{RUN}_3seeds.json"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
    if MAIN_DRIVE:
        Path(MAIN_DRIVE).mkdir(parents=True, exist_ok=True)
        shutil.copy2(summary_path, Path(MAIN_DRIVE) / summary_path.name)

In [ ]:
# 6. DOWNLOAD: package evaluation JSONs, logs, final_student.pth, and final_preds for each seed
# (excludes intermediate 25-epoch optimizer checkpoints so browser download succeeds reliably).
from google.colab import files
BASE_NAME = "ours_v5" if RUN == "ours" else mc.KIND_DIRS[RUN]
for seed in SEEDS:
    name = mc.seed_name(BASE_NAME, seed)
    directory = Path(RUNS) / name
    if directory.is_dir():
        archive = f"/content/results_{name}.zip"
        has_final_student = any(directory.rglob("final_student.pth"))
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
            for item in sorted(directory.rglob("*")):
                if not item.is_file():
                    continue
                if has_final_student and (item.name.startswith("checkpoint_epoch_") or item.name == "latest_checkpoint.pth"):
                    continue
                zf.write(item, item.relative_to(Path(RUNS)).as_posix())
        files.download(archive)
    else:
        print(f"Skipping {name}: local directory {directory} not found")